# Step 3 – Cleaning and the data quality report
Goal: dates as real dates, the fiscal year, readable channel names, and every strange thing **flagged and documented**, not deleted.
Output: `clean/cottonil_clean.pkl` (one file for all notebooks, the dashboard and the website) and `outputs/data_quality_report.csv`.

In [1]:
import sys, os
import pandas as pd
sys.path.insert(0, '..')
from core.build_clean import load_raw, clean, quality_report, fiscal_year, CHANNEL

t = load_raw('../Cottonil_project/COTTONIL/COTTONIL_Data/02_TABLES')
print(len(t), 'tables')

25 tables


## 3.1 Dates
Dates are read as text. Three tables use `M/D/YYYY` (branches, customers, employees, imports) – we give the format so 3/4 is not read as 4 March.

In [2]:
for name, col in [('sales_orders', 'order_date'), ('branches', 'opening_date'), ('import_shipments', 'shipment_date')]:
    print(f'{name}.{col}:', t[name][col].iloc[0], '→ type', t[name][col].dtype)
print(pd.to_datetime(t['branches'].opening_date, format='%m/%d/%Y').head(3).dt.date.tolist())

sales_orders.order_date: 2018-10-19 → type str
branches.opening_date: 9/14/2019 → type str
import_shipments.shipment_date: 10/1/2019 → type str
[datetime.date(2019, 9, 14), datetime.date(2019, 2, 19), datetime.date(2020, 2, 12)]


## 3.2 Fiscal year Oct–Sep (same rule as Sharnoubi)
The data runs 19 Oct 2018 → 18 Oct 2026. With calendar years, 2018 would have 2.5 months and 2026 nine and a half.
With Oct–Sep we get **8 complete years FY2019 → FY2026**, and October 2026 (18 days) is left out of yearly comparisons.

In [3]:
d = pd.to_datetime(pd.Series(['2025-09-30', '2025-10-01', '2026-09-30']))
pd.DataFrame({'date': d.dt.date, 'fiscal year': fiscal_year(d)})

,date,fiscal year
0,2025-09-30,2025
1,2025-10-01,2026
2,2026-09-30,2026


## 3.3 Run the cleaning (all tables) and save one file

In [4]:
c = clean(t)
pd.to_pickle(c, '../clean/cottonil_clean.pkl')
print('saved', len(c), 'tables:', ', '.join(c))
print('channels:', CHANNEL)

saved 25 tables: branches, customers, products, skus, employees, warehouses, suppliers, factories, import_partners, data_dictionary, public_catalog, orders, lines, payments, returns, shipments, stock, transfers, transfer_lines, pos, po_lines, imports, campaigns, payroll, opex
channels: {'Retail POS': 'Store', 'E-commerce': 'Online', 'Wholesale / reseller': 'Wholesale'}


In [5]:
# what we added to the sales lines (one row = one product in an order)
c['lines'][['sales_line_id', 'order_date', 'fy', 'channel', 'customer_id', 'category_en', 'size', 'quantity', 'net_line_EGP', 'cost_line_EGP', 'has_discount', 'discount_pct']].head()

,sales_line_id,order_date,fy,channel,customer_id,category_en,size,quantity,net_line_EGP,cost_line_EGP,has_discount,discount_pct
0,SL000000001,2018-10-19,2019,Store,C0000006,Men tops,L,1,24.30,12.95,True,0.100000
1,SL000000002,2018-10-19,2019,Store,C0000010,Men underwear,4XL,1,61.19,48.74,True,0.150021
2,SL000000003,2018-10-19,2019,Store,C0000010,Socks,M,6,207.36,100.08,False,0.000000
3,SL000000004,2018-10-19,2019,Online,NaN,Men tops,L,1,128.58,85.44,True,0.199975
4,SL000000005,2018-10-19,2019,Store,C0000025,Socks,M,1,30.46,21.09,True,0.049906


## 3.4 Checks we run before trusting any number

In [6]:
so, li = c['orders'], c['lines']
checks = pd.DataFrame([
    ('duplicate rows – all tables', sum(int(v.duplicated().sum()) for v in t.values())),
    ('order net = gross − discount (|diff| > 1 EGP)', int(((so.gross_EGP - so.discount_EGP - so.net_before_tax_EGP).abs() > 1).sum())),
    ('line net = gross − discount', int(((li.gross_line_EGP - li.discount_line_EGP - li.net_line_EGP).abs() > 1).sum())),
    ('order net = sum of its lines', int((li.groupby('sales_order_id').net_line_EGP.sum() - so.set_index('sales_order_id').net_before_tax_EGP).abs().gt(1).sum())),
    ('margin = net − cost', int(((li.net_line_EGP - li.cost_line_EGP - li.margin_EGP).abs() > 1).sum())),
    ('empty cells – sales lines (other than customer_id)', int(li.drop(columns='customer_id').isna().sum().sum())),
    ('sales lines without a customer_id', int(li.customer_id.isna().sum())),
], columns=['check', 'problem rows'])
checks

,check,problem rows
0,duplicate rows – all tables,0
1,order net = gross − discount (|diff| > 1 EGP),0
2,line net = gross − discount,0
3,order net = sum of its lines,0
4,margin = net − cost,0
5,empty cells – sales lines (other than customer_id),0
6,sales lines without a customer_id,216947


All arithmetic closes 100%. So the problems are **not typing errors** – they are columns filled without a link to reality.
## 3.5 The data quality report
Severity (same idea as Sharnoubi, plus one level):
- **Business** – not a data error: it IS a finding the company must act on.
- **High** – breaks an important analysis.
- **Medium** – a gap in the systems to fix.
- **Unreliable** – a field filled at random: we build no decision on it.

In [7]:
dq = quality_report(c)
dq.to_csv('../outputs/data_quality_report.csv', index=False, encoding='utf-8-sig')
print(dq.severity.value_counts().to_dict())
dq[['id', 'area', 'issue', 'evidence', 'severity']]

{'Unreliable': 9, 'High': 8, 'Medium': 6, 'Business': 5}


,id,area,issue,evidence,severity
0,DQ01,Purchasing,Purchased units are ~100x the units sold,"339,452,167 bought vs 3,398,655 sold (100x)",Business
1,DQ02,Purchasing,Quantity bought per SKU does not follow its sales,correlation per SKU = 0.011,Business
2,DQ03,Distribution,Branches receive far more than they sell,"12,677,158 units sent vs 3,398,655 sold (3.7x)",Business
3,DQ04,Distribution,Branches receive goods from the raw-material and returns warehouses,17% of units sent to branches,High
4,DQ05,Sales,Discounts do not increase the quantity sold,54% of lines discounted; avg qty 2.99 vs 2.98,Business
5,DQ06,Marketing,Campaigns claim far more orders than exist,"24,106,747 attributed vs 540,000 real orders (44.6x)",High
6,DQ07,Branches,Orders dated before the branch opened,"38,040 orders (7.0%), 18.3M EGP",Medium
7,DQ08,Branches,All branches sell almost the same,"store orders per branch 4,699 – 5,011",Unreliable
8,DQ09,Delivery,"""Delayed"" status is not linked to delivery time","428 'Delayed' delivered in ≤1 day; 13,512 'on time' took 7 days",Unreliable
9,DQ10,Delivery,Delivery time is the same for every governorate,3.93 – 4.05 days (Cairo = Aswan),Unreliable


**Takeaway:** 28 issues. 5 are business findings (purchasing, transfers, discounts, free shipping),
9 are fields we will not use for decisions, and the rest are gaps the company must fix (customer ID, payroll, HR roles, stock count).